# 01 - Train the Segmentation Head (SIIM-ACR Pneumothorax)

Trains DenseNet-121 encoder + segmentation decoder (Tiramisu-style, with skip connections) end-to-end, using the Dice loss. Encoder starts from ImageNet weights.

In [ ]:
import os
import sys

REPO_NAME = "multi-task-chest-xray-analysis-system"
PROJECT_DIR = f"/kaggle/working/{REPO_NAME}"

assert os.path.exists(PROJECT_DIR), f"{PROJECT_DIR} does not exist!"

sys.path.insert(0, PROJECT_DIR)

print("Project directory:", PROJECT_DIR)

import torch
from src import config, engine
from src.models.mtl_model import MultiCheXNet
from src.datasets.siim_acr import (SIIMACRDataset, build_siim_dataframe,
                                    get_default_train_augmentation, train_val_split)
from src.utils.visualize import show_image_mask, show_training_curves

print("device:", config.DEVICE)


## 1. Point these at your downloaded data (see notebook 00)

In [ ]:
SEG_CSV_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data/train-rle.csv"
SEG_IMG_PATH = "/kaggle/input/datasets/jesperdramsch/siim-acr-pneumothorax-segmentation-data"
DET_CSV_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"
DET_IMG_PATH = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images"


# only_positive=True matches the upstream reference repo's default
# (coursat-ai/MultiCheXNet). ~80% of SIIM-ACR images have an entirely empty
# pneumothorax mask; training on that mix gives Dice loss a trivial
# "predict nothing" shortcut that it collapses into almost immediately (see
# notebook discussion / earlier debugging in this project). Restricting
# training to images that actually contain a lesion removes that shortcut
# at the source rather than patching around it.
df = build_siim_dataframe(SEG_CSV_PATH, SEG_IMG_PATH, only_positive=True)
print(len(df), "rows,", df['ImageId'].nunique(), "unique positive images")
df.head()


In [ ]:
train_ids, val_ids = train_val_split(df, val_fraction=0.15)
print(len(train_ids), "train images,", len(val_ids), "val images")
# (df is already only_positive=True, so every image here has a real mask -
# no separate positive/negative balancing needed anymore; plain shuffling
# is fine.)

train_ds = SIIMACRDataset(df, train_ids, augmentation=get_default_train_augmentation())
val_ds = SIIMACRDataset(df, val_ids, augmentation=None)

BATCH_SIZE = 16   # T4 16GB handles this comfortably at 256x256
from torch.utils.data import DataLoader
# num_workers=2: DICOM decode + resize is CPU-bound, and parallel workers
# matter a lot for this dataset's per-item load time. You may see a harmless
# "AssertionError: can only test a child process" printed during worker
# cleanup under Kaggle/Jupyter's forked multiprocessing - it happens after
# each epoch finishes and does NOT affect training; it's cosmetic only.
# persistent_workers=True avoids re-spawning the worker pool every epoch.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, persistent_workers=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, persistent_workers=True)


## 2. Peek at a training sample

In [ ]:
img, mask, label = train_ds[0]
show_image_mask(img.permute(1,2,0).numpy(), mask[0].numpy(), title=f"class={label}")


## 3. Build the model (segmentation head only) and train

In [ ]:
model = MultiCheXNet(pretrained_encoder=True, add_classifier=False,
                     add_detector=False, add_segmenter=True).to(config.DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=(config.DEVICE == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3)

EPOCHS = 25
history = engine.fit(
    engine.train_one_epoch_segmenter, engine.evaluate_segmenter,
    model, train_loader, val_loader, optimizer, EPOCHS,
    device=config.DEVICE, checkpoint_path="/kaggle/working/multi-task-chest-xray-analysis-system/segmenter_best.pt",
    # pos_dice = dice averaged only over images that actually contain
    # pneumothorax. Since df/train_ds/val_ds are now only_positive=True,
    # every image qualifies, so pos_dice ~= dice here - it's kept mainly for
    # consistency with notebook 04 (joint MTL), where the segmentation
    # source is also only_positive but the overall val loader/history mixes
    # in detection-only batches.
    # pos_weight="auto": computed per-batch from the actual pixel-level
    # imbalance (a lesion is often only 3-10% of a positive image's pixels,
    # which a fixed pos_weight badly under- or over-shoots depending on the
    # batch) - see BCEDiceLoss docstring in src/losses/seg_loss.py.
    monitor="pos_dice", mode="max", scaler=scaler,
    scheduler=scheduler, patience=7, pos_weight="auto",
)


## 4. Training curves & qualitative results

In [ ]:
show_training_curves({k: v for k, v in history.items() if 'loss' in k})
show_training_curves({k: v for k, v in history.items() if k in ('train_dice','val_dice')})
show_training_curves({k: v for k, v in history.items() if 'pos_dice' in k})


In [ ]:
model.load("/kaggle/working/multi-task-chest-xray-analysis-system/segmenter_best.pt", map_location=config.DEVICE)
model.eval()

img, mask, label = val_ds[0]
with torch.no_grad():
    pred = model(img.unsqueeze(0).to(config.DEVICE))["seg"][0,0].cpu().numpy()

show_image_mask(img.permute(1,2,0).numpy(), mask[0].numpy(), pred_mask=pred>0.5)


Checkpoint saved to `/content/segmenter_best.pt` - this file only contains the segmentation head + encoder weights (since `add_classifier=False, add_detector=False`). It will be reused (its *encoder* weights) as a warm start in `04_Train_MTL_Joint.ipynb`.